In [ ]:
# Install dependencies (Kaggle-only; skip if baked into image)
%pip install -q -U datasets transformers accelerate evaluate torch

In [ ]:
import os, json, random, pathlib
from datasets import load_from_disk, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer
from transformers.trainer_utils import IntervalStrategy
import evaluate

DATA_DIR = pathlib.Path('/kaggle/input/ura-pdf-chunks')  # replace with your dataset path
MODEL_NAME = os.environ.get('BASE_MODEL', 'google/flan-t5-small')
OUTPUT_DIR = pathlib.Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# Expect dataset with columns: 'question', 'context', 'answer'
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Dataset dir not found: {DATA_DIR}')
raw = load_from_disk(str(DATA_DIR))
if not isinstance(raw, DatasetDict):
    raw = DatasetDict({'train': raw.train_test_split(test_size=0.1)['train'], 'validation': raw.train_test_split(test_size=0.1)['test']})
print(raw)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    inputs = [f'question: {q} context: {c}' for q, c in zip(batch['question'], batch['context'])]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(batch['answer'], max_length=128, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized = raw.map(preprocess, batched=True, remove_columns=list(raw['train'].features))
tokenized

In [ ]:
metric = evaluate.load('sacrebleu')

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = tokenizer.batch_decode([[l for l in label if l != -100] for label in labels], skip_special_tokens=True)
    return {'sacrebleu': metric.compute(predictions=preds, references=[[l] for l in labels])['score']}

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    evaluation_strategy=IntervalStrategy.STEPS,
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    predict_with_generate=True,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR / 'final-model'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'final-model'))

## Notes
- Replace `DATA_DIR` with your processed URA dataset location on Kaggle (e.g., /kaggle/input/...).
- Tune hyperparameters (batch size, epochs, lr).
- For larger models, enable bf16/flash attention if hardware supports.
- Upload `outputs/final-model` as a Kaggle dataset or GitHub Release for deployment.